# ORB-BSCNN for Heart Disease Diagnosis — reproducible experiments

One notebook that produces every table in the manuscript from a single run.

**What is different from the earlier notebooks**

- The random seed is reset before every model in every fold, so each number is reproducible and does not depend on which cells were run before.
- Cross-validation is repeated (5 repeats x 5 folds = 25 paired folds), and the significance tests use the same per-fold values that the results table is built from.
- The correlation-threshold table and the main results table share seeds, so the row for the default threshold is identical to the main result.
- Multi-site analysis: missing values are imputed inside each training fold, and a leave-one-site-out evaluation is added.
- Added a logistic-regression baseline and three ablations (no binarization, no residual connection, depth-matched patient graph).
- Everything is written to `ORB_BSCNN_results.xlsx`.

**How to run:** Runtime -> Run all. It runs on CPU on purpose (the data are small, and CPU runs are repeatable). Expect roughly 10-20 minutes.

## 1. Setup and configuration

In [ ]:
import numpy as np, pandas as pd, networkx as nx
import random, time, sys, platform
import sklearn, scipy
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

BASE_SEED      = 42
N_REPEATS      = 5      # repeats of 5-fold CV -> 25 paired folds
N_SPLITS       = 5
SWEEP_REPEATS  = 2      # repeats used for the hyperparameter sweep (kept small for runtime)
EPOCHS         = 150
CORR_THRESHOLD = 0.15   # default correlation threshold for the feature graph
K_NEIGHBOURS   = 8      # default k for the patient graph
THRESHOLDS     = [0.05, 0.10, 0.15, 0.20, 0.30]
METRICS        = ["Accuracy", "Sensitivity", "Specificity", "F1", "AUC"]

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cpu")          # CPU on purpose: small data, repeatable results
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

versions = pd.DataFrame({"package": ["python", "torch", "scikit-learn", "numpy", "pandas", "networkx", "scipy"],
                         "version": [platform.python_version(), torch.__version__, sklearn.__version__,
                                     np.__version__, pd.__version__, nx.__version__, scipy.__version__]})
versions

In [ ]:
MODEL_IDS = {"ORB-BSCNN (patient graph)": 1, "ORB-BSCNN (feature graph)": 2, "MLP": 3,
             "Random Forest": 4, "Logistic Regression": 5,
             "Ablation: feature graph, no binarization": 2,   # same seed as the full model, so only the component differs
             "Ablation: feature graph, no residual": 2,
             "Ablation: patient graph, 2 blocks": 1}

def run_seed(repeat, fold, model_name):
    """One fixed seed per (repeat, fold, model)."""
    return BASE_SEED + 10000 * repeat + 100 * fold + MODEL_IDS[model_name]

## 2. Data

In [ ]:
COLS = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
        "exang","oldpeak","slope","ca","thal","target"]
FEATURE_COLS = COLS[:-1]
BASE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/"
SOURCES = {"cleveland": "processed.cleveland.data", "hungarian": "processed.hungarian.data",
           "switzerland": "processed.switzerland.data", "va": "processed.va.data"}

site_frames = {}
for site, fname in SOURCES.items():
    d = pd.read_csv(BASE_URL + fname, names=COLS, na_values="?")
    d["source"] = site
    site_frames[site] = d
    print(f"{site:12s}: {d.shape[0]} rows, {int(d[FEATURE_COLS].isnull().sum().sum())} missing values")

# Cleveland, complete cases only (primary dataset)
df = site_frames["cleveland"].dropna().reset_index(drop=True)
table1_sample = df[COLS].sample(5, random_state=BASE_SEED)      # raw records, original UCI coding
df["target"] = (df["target"] > 0).astype(int)
X = df[FEATURE_COLS].values.astype(np.float32)
y = df["target"].values.astype(np.int64)
print("\nCleveland complete cases:", X.shape, " class counts:", np.bincount(y))

# Four-site collection, missing values kept (imputed later inside each training fold)
df_full = pd.concat(site_frames.values(), ignore_index=True)
df_full["target"] = (df_full["target"] > 0).astype(int)
X_full = df_full[FEATURE_COLS].values.astype(np.float32)
y_full = df_full["target"].values.astype(np.int64)
site_full = df_full["source"].values
print("Four-site collection:", X_full.shape, " class counts:", np.bincount(y_full))
table1_sample

## 3. Simplicial complex construction

Both complexes are built with boundary matrices B1 (node-edge) and B2 (edge-triangle).
The model propagates node signals with the symmetric-normalized 0-th order Hodge Laplacian L0 = B1 B1^T, so B2 (the triangle structure) is built and reported but does not enter the forward pass.

In [ ]:
def _complex_from_graph(G, n_nodes, with_B2=False):
    edges = [tuple(sorted(e)) for e in G.edges()]
    B1 = np.zeros((n_nodes, len(edges)), dtype=np.float32)
    for j, (u, v) in enumerate(edges):
        B1[u, j] = -1.0; B1[v, j] = 1.0
    triangles = []
    for c in nx.enumerate_all_cliques(G):        # yields cliques in order of size
        if len(c) > 3: break
        if len(c) == 3: triangles.append(tuple(sorted(c)))
    cplx = {"n_nodes": n_nodes, "n_edges": len(edges), "n_tri": len(triangles), "B1": B1, "edges": edges}
    if with_B2:
        edge_index = {e: i for i, e in enumerate(edges)}
        B2 = np.zeros((len(edges), len(triangles)), dtype=np.float32)
        for t, (a, b, c) in enumerate(triangles):
            B2[edge_index[(a, b)], t] = 1.0; B2[edge_index[(b, c)], t] = 1.0; B2[edge_index[(a, c)], t] = -1.0
        cplx["B2"] = B2
    return cplx

def build_patient_complex(Xd, k=K_NEIGHBOURS, with_B2=False):
    """Nodes = patients, edges = k nearest neighbours in standardized feature space."""
    knn = kneighbors_graph(Xd, n_neighbors=k, mode="connectivity", include_self=False)
    G = nx.Graph(nx.from_scipy_sparse_array(knn))
    return _complex_from_graph(G, Xd.shape[0], with_B2)

def build_feature_complex(X_tr, corr_threshold=CORR_THRESHOLD, with_B2=False):
    """Nodes = features, edges = |Pearson r| >= threshold, computed on training data only."""
    corr = np.nan_to_num(np.corrcoef(X_tr, rowvar=False))
    n_feat = corr.shape[0]
    G = nx.Graph(); G.add_nodes_from(range(n_feat))
    for i in range(n_feat):
        for j in range(i + 1, n_feat):
            if abs(corr[i, j]) >= corr_threshold:
                G.add_edge(i, j)
    return _complex_from_graph(G, n_feat, with_B2)

def normalized_L0(cplx):
    L = cplx["B1"] @ cplx["B1"].T
    d = np.diag(L).copy(); d[d == 0] = 1.0
    D = np.diag(1.0 / np.sqrt(d))
    return D @ L @ D

def full_metrics(y_true, preds, probs):
    tn, fp, fn, tp = confusion_matrix(y_true, preds, labels=[0, 1]).ravel()
    return {"Accuracy": accuracy_score(y_true, preds),
            "Sensitivity": tp / (tp + fn) if (tp + fn) else np.nan,
            "Specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "F1": f1_score(y_true, preds, zero_division=0),
            "AUC": roc_auc_score(y_true, probs) if len(np.unique(y_true)) == 2 else np.nan}

# Feature graph on the full Cleveland set (for the figure caption)
_fc = build_feature_complex(StandardScaler().fit_transform(X), CORR_THRESHOLD, with_B2=True)
print(f"Feature graph at threshold {CORR_THRESHOLD} (all Cleveland complete cases): "
      f"{_fc['n_nodes']} nodes, {_fc['n_edges']} edges, {_fc['n_tri']} triangles")
feature_graph_edges = pd.DataFrame([(FEATURE_COLS[u], FEATURE_COLS[v]) for u, v in _fc["edges"]], columns=["feature_a", "feature_b"])

## 4. Models

Same architecture as the earlier notebooks. Only the activations are binarized (sign function with a straight-through estimator); the weights are full precision.
Two switches (`use_binarize`, `use_residual`) are added for the ablations.

In [ ]:
class BinarizeSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return torch.sign(x + 1e-12)
    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        grad = grad_output.clone()
        grad[x.abs() > 1] = 0
        return grad
binarize = BinarizeSTE.apply


class BiSimplicialConv(nn.Module):
    """Transductive version: X is [n_nodes, dim] over one fixed patient graph."""
    def __init__(self, in_dim, out_dim, laplacian, k_hops=2):
        super().__init__()
        self.register_buffer("L", laplacian)
        self.k_hops = k_hops
        self.theta = nn.Parameter(torch.randn(k_hops + 1, in_dim, out_dim) * 0.1)
        self.bn = nn.BatchNorm1d(out_dim)
    def forward(self, X):
        out = 0; Lx = X
        for h in range(self.k_hops + 1):
            out = out + Lx @ self.theta[h]
            Lx = self.L @ Lx
        return self.bn(out)


class BatchedBiSimplicialConv(nn.Module):
    """Batched version: X is [batch, n_nodes, dim] over one fixed feature graph."""
    def __init__(self, in_dim, out_dim, laplacian, k_hops=2):
        super().__init__()
        self.register_buffer("L", laplacian)
        self.k_hops = k_hops
        self.theta = nn.Parameter(torch.randn(k_hops + 1, in_dim, out_dim) * 0.1)
        self.bn = nn.BatchNorm1d(out_dim)
    def forward(self, X):
        out = 0; Lx = X
        for h in range(self.k_hops + 1):
            out = out + Lx @ self.theta[h]
            Lx = torch.einsum("ij,bjf->bif", self.L, Lx)
        b, n, f = out.shape
        return self.bn(out.reshape(b * n, f)).reshape(b, n, f)


class ResidualBlock(nn.Module):
    def __init__(self, conv, use_binarize=True, use_residual=True):
        super().__init__()
        self.conv = conv
        self.act = nn.PReLU()
        self.use_binarize = use_binarize
        self.use_residual = use_residual
    def forward(self, X):
        pre = self.conv(X)
        h = self.act(binarize(pre) + pre) if self.use_binarize else self.act(pre)
        return h + X if self.use_residual else h


def _head(hidden_dim, n_classes=2):
    return nn.Sequential(nn.Linear(hidden_dim, hidden_dim // 2), nn.PReLU(), nn.Dropout(0.3),
                         nn.Linear(hidden_dim // 2, n_classes))

class ORB_BSCNN(nn.Module):
    """Patient-graph model (transductive)."""
    def __init__(self, in_dim, hidden_dim, laplacian, n_blocks=3, k_hops=2, use_binarize=True, use_residual=True):
        super().__init__()
        self.input_proj = nn.Linear(in_dim, hidden_dim)
        self.blocks = nn.ModuleList([ResidualBlock(BiSimplicialConv(hidden_dim, hidden_dim, laplacian, k_hops),
                                                   use_binarize, use_residual) for _ in range(n_blocks)])
        self.classifier = _head(hidden_dim)
    def forward(self, X):
        h = self.input_proj(X)
        for block in self.blocks:
            h = block(h)
        return self.classifier(h)

class FeatureGraphORB_BSCNN(nn.Module):
    """Feature-graph model (inductive): each patient is a signal over the feature graph."""
    def __init__(self, hidden_dim, laplacian, n_blocks=2, k_hops=2, use_binarize=True, use_residual=True):
        super().__init__()
        self.input_proj = nn.Linear(1, hidden_dim)
        self.blocks = nn.ModuleList([ResidualBlock(BatchedBiSimplicialConv(hidden_dim, hidden_dim, laplacian, k_hops),
                                                   use_binarize, use_residual) for _ in range(n_blocks)])
        self.classifier = _head(hidden_dim)
    def forward(self, X):
        h = self.input_proj(X.unsqueeze(-1))
        for block in self.blocks:
            h = block(h)
        return self.classifier(h.mean(dim=1))

class MLP(nn.Module):
    def __init__(self, in_dim, hidden=32, n_classes=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(),
                                 nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, n_classes))
    def forward(self, x): return self.net(x)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
criterion = nn.CrossEntropyLoss()
_t = lambda a, dt=torch.float32: torch.tensor(a, dtype=dt, device=device)

def _fit(model, forward_train, y_train_t, lr, epochs, adamw=True):
    """Full-batch training. The scheduler monitors the TRAINING loss; no test data are used."""
    if adamw:
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=15)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=lr); sched = None
    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        loss = criterion(forward_train(), y_train_t)
        loss.backward(); opt.step()
        if sched is not None: sched.step(loss.item())
    model.eval()

def _predict(logits, y_te):
    probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
    preds = logits.argmax(dim=1).cpu().numpy()
    return full_metrics(y_te, preds, probs)

def train_patient_graph(X_tr, y_tr, X_te, y_te, k=K_NEIGHBOURS, hidden_dim=32, n_blocks=3, k_hops=2,
                        lr=1e-2, epochs=EPOCHS, use_binarize=True, use_residual=True, return_model=False):
    """Transductive: the graph contains training and test patients; only training labels are used."""
    X_a = np.vstack([X_tr, X_te]); n_tr = X_tr.shape[0]
    L = _t(normalized_L0(build_patient_complex(X_a, k=k)))
    Xat = _t(X_a); ytr = _t(y_tr, torch.long)
    m = ORB_BSCNN(X_a.shape[1], hidden_dim, L, n_blocks, k_hops, use_binarize, use_residual).to(device)
    _fit(m, lambda: m(Xat)[:n_tr], ytr, lr, epochs)
    with torch.no_grad():
        res = _predict(m(Xat)[n_tr:], y_te)
    return (res, m, Xat) if return_model else res

def train_feature_graph(X_tr, y_tr, X_te, y_te, corr_threshold=CORR_THRESHOLD, hidden_dim=32, n_blocks=2, k_hops=2,
                        lr=1e-2, epochs=EPOCHS, use_binarize=True, use_residual=True, return_model=False):
    L = _t(normalized_L0(build_feature_complex(X_tr, corr_threshold)))
    Xtr = _t(X_tr); ytr = _t(y_tr, torch.long); Xte = _t(X_te)
    m = FeatureGraphORB_BSCNN(hidden_dim, L, n_blocks, k_hops, use_binarize, use_residual).to(device)
    _fit(m, lambda: m(Xtr), ytr, lr, epochs)
    with torch.no_grad():
        res = _predict(m(Xte), y_te)
    return (res, m, Xte) if return_model else res

def train_mlp(X_tr, y_tr, X_te, y_te, epochs=EPOCHS, return_model=False):
    Xtr = _t(X_tr); ytr = _t(y_tr, torch.long); Xte = _t(X_te)
    m = MLP(X_tr.shape[1]).to(device)
    _fit(m, lambda: m(Xtr), ytr, 1e-3, epochs, adamw=False)
    with torch.no_grad():
        res = _predict(m(Xte), y_te)
    return (res, m, Xte) if return_model else res

In [ ]:
def train_rf(X_tr, y_tr, X_te, y_te, seed=BASE_SEED):
    rf = RandomForestClassifier(n_estimators=300, random_state=seed).fit(X_tr, y_tr)
    return full_metrics(y_te, rf.predict(X_te), rf.predict_proba(X_te)[:, 1])

def train_logreg(X_tr, y_tr, X_te, y_te):
    lr_ = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    return full_metrics(y_te, lr_.predict(X_te), lr_.predict_proba(X_te)[:, 1])

MODEL_FUNCS = {
    "ORB-BSCNN (patient graph)": lambda a, b, c, d: train_patient_graph(a, b, c, d),
    "ORB-BSCNN (feature graph)": lambda a, b, c, d: train_feature_graph(a, b, c, d),
    "MLP":                       lambda a, b, c, d: train_mlp(a, b, c, d),
    "Random Forest":             lambda a, b, c, d: train_rf(a, b, c, d),
    "Logistic Regression":       lambda a, b, c, d: train_logreg(a, b, c, d),
    "Ablation: feature graph, no binarization": lambda a, b, c, d: train_feature_graph(a, b, c, d, use_binarize=False),
    "Ablation: feature graph, no residual":     lambda a, b, c, d: train_feature_graph(a, b, c, d, use_residual=False),
    "Ablation: patient graph, 2 blocks":        lambda a, b, c, d: train_patient_graph(a, b, c, d, n_blocks=2),
}
MAIN_MODELS = ["ORB-BSCNN (patient graph)", "ORB-BSCNN (feature graph)", "MLP", "Random Forest", "Logistic Regression"]
ABLATIONS   = ["Ablation: feature graph, no binarization", "Ablation: feature graph, no residual",
               "Ablation: patient graph, 2 blocks"]

def prepare_fold(Xtr_raw, Xte_raw):
    """Median imputation and standardization, both fitted on the training fold only."""
    med = np.nanmedian(Xtr_raw, axis=0)
    med = np.where(np.isnan(med), 0.0, med)
    Xtr = np.where(np.isnan(Xtr_raw), med, Xtr_raw)
    Xte = np.where(np.isnan(Xte_raw), med, Xte_raw)
    sc = StandardScaler().fit(Xtr)
    return sc.transform(Xtr).astype(np.float32), sc.transform(Xte).astype(np.float32)

def run_models(Xtr, ytr, Xte, yte, repeat, fold, models):
    out = {}
    for name in models:
        set_seed(run_seed(repeat, fold, name))
        out[name] = MODEL_FUNCS[name](Xtr, ytr, Xte, yte)
    return out

def repeated_cv(Xd, yd, models, n_repeats=N_REPEATS, label=""):
    rows = []; t0 = time.time()
    for rep in range(n_repeats):
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=BASE_SEED + rep)
        for fold, (tr, te) in enumerate(skf.split(Xd, yd)):
            Xtr, Xte = prepare_fold(Xd[tr], Xd[te])
            res = run_models(Xtr, yd[tr], Xte, yd[te], rep, fold, models)
            for name, m in res.items():
                rows.append({"repeat": rep, "fold": fold, "model": name, **m})
        print(f"{label} repeat {rep + 1}/{n_repeats} done ({time.time() - t0:.0f}s)")
    return pd.DataFrame(rows)

def summarize(per_fold, order=None):
    g = per_fold.groupby("model")[METRICS].agg(["mean", "std"])
    if order: g = g.loc[[m for m in order if m in g.index]]
    g.columns = [f"{m}_{s}" for m, s in g.columns]
    return g.round(4)

## 5. Cleveland: repeated 5-fold cross-validation (main results and ablations)

In [ ]:
cv_cleveland = repeated_cv(X, y, MAIN_MODELS + ABLATIONS, label="Cleveland")
table2 = summarize(cv_cleveland, MAIN_MODELS)
table_ablation = summarize(cv_cleveland, ["ORB-BSCNN (feature graph)", "Ablation: feature graph, no binarization",
                                          "Ablation: feature graph, no residual",
                                          "ORB-BSCNN (patient graph)", "Ablation: patient graph, 2 blocks"])
print("Main results (mean and SD over", N_REPEATS * N_SPLITS, "folds)")
table2

In [ ]:
table_ablation

## 6. Significance tests on the same per-fold values

Three tests per comparison, all paired on the 25 matched folds:
the corrected resampled t-test (Nadeau and Bengio), which accounts for the overlap between training sets and is the one to report;
the ordinary paired t-test (too optimistic for repeated CV, shown for reference);
and the Wilcoxon signed-rank test.

In [ ]:
def paired_tests(per_fold, model_a, model_b, metric, test_train_ratio=1.0 / (N_SPLITS - 1)):
    wide = per_fold.pivot_table(index=["repeat", "fold"], columns="model", values=metric)
    a, b = wide[model_a].values, wide[model_b].values
    d = a - b; J = len(d); var = d.var(ddof=1)
    if var > 0:
        t_corr = d.mean() / np.sqrt((1.0 / J + test_train_ratio) * var)
        p_corr = 2 * stats.t.sf(abs(t_corr), J - 1)
        t_naive, p_naive = stats.ttest_rel(a, b)
    else:
        t_corr = p_corr = t_naive = p_naive = np.nan
    try:
        _, p_w = stats.wilcoxon(a, b)
    except ValueError:
        p_w = np.nan
    return {"Comparison": f"{model_a} vs. {model_b}", "Metric": metric, "Mean A": a.mean(), "Mean B": b.mean(),
            "Mean diff": d.mean(), "Folds A>B": int((d > 0).sum()), "Folds tied": int((d == 0).sum()), "Folds": J,
            "Corrected t": t_corr, "Corrected p": p_corr, "Paired t": t_naive, "Paired t p": p_naive, "Wilcoxon p": p_w}

FG, PG = "ORB-BSCNN (feature graph)", "ORB-BSCNN (patient graph)"
comparisons = [(FG, PG, "AUC"), (FG, PG, "Sensitivity"), (FG, PG, "Accuracy"),
               (FG, "Random Forest", "AUC"), (FG, "MLP", "AUC"), (FG, "Logistic Regression", "AUC"),
               (FG, "Ablation: feature graph, no binarization", "AUC"),
               (FG, "Ablation: feature graph, no residual", "AUC"),
               (FG, "Ablation: patient graph, 2 blocks", "AUC"),
               (PG, "Ablation: patient graph, 2 blocks", "AUC")]
table8 = pd.DataFrame([paired_tests(cv_cleveland, a, b, m) for a, b, m in comparisons]).round(4)
table8

## 7. Correlation-threshold ablation (feature graph)

Uses the same folds and seeds as Section 5, so the row for the default threshold equals the main result.

In [ ]:
rows = []
for th in THRESHOLDS:
    for rep in range(N_REPEATS):
        skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=BASE_SEED + rep)
        for fold, (tr, te) in enumerate(skf.split(X, y)):
            Xtr, Xte = prepare_fold(X[tr], X[te])
            fc = build_feature_complex(Xtr, th)
            set_seed(run_seed(rep, fold, FG))
            m = train_feature_graph(Xtr, y[tr], Xte, y[te], corr_threshold=th)
            rows.append({"threshold": th, "repeat": rep, "fold": fold, "n_edges": fc["n_edges"], "n_triangles": fc["n_tri"], **m})
    print(f"threshold {th} done")
cv_threshold = pd.DataFrame(rows)
g = cv_threshold.groupby("threshold")[METRICS + ["n_edges", "n_triangles"]].agg(["mean", "std"])
g.columns = [f"{m}_{s}" for m, s in g.columns]
table4 = g.round(4)
table4

## 8. Hyperparameter sensitivity (patient graph)

One hyperparameter is varied at a time. Each setting is scored by cross-validation, not on a single test split.

In [ ]:
grid = {"k": [5, 8, 12], "hidden_dim": [16, 32, 64], "n_blocks": [1, 2, 3], "k_hops": [1, 2], "lr": [1e-2, 5e-3]}
defaults = {"k": 8, "hidden_dim": 32, "n_blocks": 3, "k_hops": 2, "lr": 1e-2}
rows = []
for pname, values in grid.items():
    for v in values:
        cfg = dict(defaults); cfg[pname] = v
        for rep in range(SWEEP_REPEATS):
            skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=BASE_SEED + rep)
            for fold, (tr, te) in enumerate(skf.split(X, y)):
                Xtr, Xte = prepare_fold(X[tr], X[te])
                set_seed(run_seed(rep, fold, PG))
                m = train_patient_graph(Xtr, y[tr], Xte, y[te], **cfg)
                rows.append({"swept_param": pname, "value": v, **m})
        print(f"swept {pname} = {v}")
cv_sweep = pd.DataFrame(rows)
g = cv_sweep.groupby(["swept_param", "value"])[["Accuracy", "AUC"]].agg(["mean", "std"])
g.columns = [f"{m}_{s}" for m, s in g.columns]
table3 = g.round(4)
table3

## 9. Model size and inference time

All timings are on CPU. The feature-graph model and the MLP are timed on the same test fold.
The patient-graph model is transductive, so one pass covers every patient in the graph; time per patient is also given.

In [ ]:
def measure_latency(fn, n_runs=200, warmup=20):
    with torch.no_grad():
        for _ in range(warmup): fn()
        t0 = time.perf_counter()
        for _ in range(n_runs): fn()
    return (time.perf_counter() - t0) / n_runs * 1000.0

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=BASE_SEED)
tr, te = next(iter(skf.split(X, y)))
Xtr, Xte = prepare_fold(X[tr], X[te])
rows = []
for name, fn in [(PG, train_patient_graph), (FG, train_feature_graph), ("MLP", train_mlp)]:
    set_seed(run_seed(0, 0, name))
    _, model, x_in = fn(Xtr, y[tr], Xte, y[te], return_model=True)
    ms = measure_latency(lambda: model(x_in))
    rows.append({"Model": name, "Trainable parameters": count_params(model),
                 "Patients per pass": x_in.shape[0], "ms per pass": ms, "ms per patient": ms / x_in.shape[0]})
table5 = pd.DataFrame(rows).round(4)
table5

## 10. Four-site UCI collection

Two evaluations: pooled repeated cross-validation (imputation fitted inside each training fold), and leave-one-site-out, which tests generalization to a hospital the model has not seen.

In [ ]:
cv_multisite = repeated_cv(X_full, y_full, MAIN_MODELS, label="Four-site")
table6 = summarize(cv_multisite, MAIN_MODELS)
table6

In [ ]:
rows = []
for i, site in enumerate(SOURCES):
    te = site_full == site; tr = ~te
    Xtr, Xte = prepare_fold(X_full[tr], X_full[te])
    res = run_models(Xtr, y_full[tr], Xte, y_full[te], 0, 50 + i, MAIN_MODELS)
    for name, m in res.items():
        rows.append({"held_out_site": site, "n_test": int(te.sum()), "n_test_positive": int(y_full[te].sum()), "model": name, **m})
    print(f"held-out site {site} done")
table_loso = pd.DataFrame(rows).round(4)
table_loso

## 11. Save all results

In [ ]:
out = "ORB_BSCNN_results.xlsx"
settings = pd.DataFrame({"setting": ["BASE_SEED", "N_REPEATS", "N_SPLITS", "SWEEP_REPEATS", "EPOCHS", "CORR_THRESHOLD", "K_NEIGHBOURS",
                                     "feature graph nodes/edges/triangles (all Cleveland)"],
                         "value": [BASE_SEED, N_REPEATS, N_SPLITS, SWEEP_REPEATS, EPOCHS, CORR_THRESHOLD, K_NEIGHBOURS,
                                   f"{_fc['n_nodes']}/{_fc['n_edges']}/{_fc['n_tri']}"]})
with pd.ExcelWriter(out) as xw:
    settings.to_excel(xw, sheet_name="Settings", index=False)
    versions.to_excel(xw, sheet_name="Versions", index=False)
    table1_sample.to_excel(xw, sheet_name="Table1_sample_records")
    table2.to_excel(xw, sheet_name="Table2_Cleveland_CV")
    table_ablation.to_excel(xw, sheet_name="Ablations")
    table8.to_excel(xw, sheet_name="Table8_significance", index=False)
    table4.to_excel(xw, sheet_name="Table4_threshold")
    table3.to_excel(xw, sheet_name="Table3_hyperparameters")
    table5.to_excel(xw, sheet_name="Table5_efficiency", index=False)
    table6.to_excel(xw, sheet_name="Table6_multisite_CV")
    table_loso.to_excel(xw, sheet_name="Leave_one_site_out", index=False)
    feature_graph_edges.to_excel(xw, sheet_name="Feature_graph_edges", index=False)
    cv_cleveland.to_excel(xw, sheet_name="PerFold_Cleveland", index=False)
    cv_multisite.to_excel(xw, sheet_name="PerFold_multisite", index=False)
    cv_threshold.to_excel(xw, sheet_name="PerFold_threshold", index=False)
print("Saved", out)
try:
    from google.colab import files
    files.download(out)
except Exception:
    pass